# Query a LEAK DB for certain domains

In [ ]:
# Import dependancies
import sqlite3
import jupyter_beeper

In [ ]:
# Constants
IN_DIR="../LEAK_DB"
OUT_DIR="../OUT"
CASE="DIVD-2025-00041"
SUB="batch1"
IN_DB=f"{IN_DIR}/{CASE}-{SUB}.sqlite3"
OUT_FILE=f"{OUT_DIR}/{CASE}-{SUB}-extract.csv"

In [ ]:
!ls $IN_DIR
!ls -l $IN_DB

Add the domains that where requested below

In [ ]:
# Set up domains
domains = """
detwentsezorgcentra.nl 
dtzc.nl 
""".split()

In [ ]:
domains

In [ ]:
conn = sqlite3.connect(IN_DB)
chunk = 10000
count = 0
old_count = -1
outfile = open(OUT_FILE, "w")
outfile.write('"username", "masked_passwd", "email_apex", "url", "url_apex", "ts_found", "ts_leaked","extra_data"\n')
while old_count != count: 
    old_count = count
    for row in conn.execute(f"""
        SELECT username, masked_passwd, email_apex, url, url_apex, ts_found, ts_leaked, extra_data
        FROM   entity
        WHERE  email_apex COLLATE NOCASE IN ( "{ '", "'.join(domains) }" )
           OR  url_apex COLLATE NOCASE IN ( "{ '", "'.join(domains) }" )
        LIMIT  { chunk }
        OFFSET { count }
    """).fetchall() :
        first = True
        for f in row:
            if not first :
                outfile.write(",")
            else:
                first = False
            outfile.write('"')
            outfile.write(str(f).replace('"','""'))
            outfile.write('"')
        outfile.write("\n")
        count = count + 1
    print(f"{count:,}", end="\r")
    if count < chunk :
        old_count = count
print(f"{count:,}")
outfile.close()
beep = jupyter_beeper.Beeper()
beep.beep()

In [ ]:
!ls $OUT_DIR
!ls -l $OUT_FILE

In [ ]:
beep = jupyter_beeper.Beeper()
beep.beep()